# 01 - EDA de Ventas leyendo desde MinIO (Bronze/Silver/Gold)

Este notebook corre en el contenedor de **JupyterLab** y NO en Spark.

Objetivos:
- Conectarse directamente a MinIO (servicio `minio:9000`).
- Leer muestras de datos de:
  - `bronze/smart_inventory/ventas_procesadas`
  - `silver/smart_inventory/ventas_limpias`
  - `gold/smart_inventory/dataset_features`
- Hacer un EDA sencillo con `pandas`.


In [ ]:
# Instalación de dependencias necesarias en el contenedor de Jupyter
%pip install minio pandas pyarrow matplotlib --quiet
import io
import pandas as pd
from minio import Minio
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8')

print("Versiones:")
print("pandas:", pd.__version__)

In [ ]:
# Configuración de conexión a MinIO
# Si cambiaste usuario/clave en tu .env, actualiza aquí.
MINIO_ENDPOINT = "minio:9000"  # nombre del servicio en docker-compose
MINIO_ACCESS_KEY = "admin"
MINIO_SECRET_KEY = "admin123"
MINIO_SECURE = False  # HTTP dentro de la red docker

client = Minio(
    MINIO_ENDPOINT,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=MINIO_SECURE,
)

print("Conectado a MinIO en:", MINIO_ENDPOINT)

In [ ]:
# Helper para leer algunos archivos Parquet desde un bucket/prefijo
def leer_parquet_minio(bucket: str, prefix: str, max_files: int = 5) -> pd.DataFrame:
    """Lee hasta max_files objetos .parquet desde MinIO y los concatena en un DataFrame."""
    objetos = client.list_objects(bucket, prefix=prefix, recursive=True)
    dfs = []
    for obj in objetos:
        if not obj.object_name.endswith(".parquet"):
            continue
        print(f"Leyendo {bucket}/{obj.object_name} ...")
        data = client.get_object(bucket, obj.object_name).read()
        df = pd.read_parquet(io.BytesIO(data))
        dfs.append(df)
        if len(dfs) >= max_files:
            break
    if not dfs:
        raise ValueError(f"No se encontraron archivos Parquet en {bucket}/{prefix}")
    return pd.concat(dfs, ignore_index=True)

# Definimos paths lógicos (bucket + prefijo) equivalentes a tus rutas s3a://
BRONZE_BUCKET = "bronze"
BRONZE_PREFIX = "smart_inventory/ventas_procesadas"

SILVER_BUCKET = "silver"
SILVER_PREFIX = "smart_inventory/ventas_limpias"

GOLD_BUCKET = "gold"
GOLD_PREFIX_FEATURES = "smart_inventory/dataset_features"

## 1. Muestra de Bronze (`ventas_procesadas`)

Datos recién ingresados desde Landing a Bronze, guardados como Parquet en MinIO.


In [ ]:
df_bronce = leer_parquet_minio(BRONZE_BUCKET, BRONZE_PREFIX, max_files=3)
print("Filas en la muestra de Bronze:", len(df_bronce))
display(df_bronce.head())
display(df_bronce.dtypes)

## 2. Muestra de Silver (`ventas_limpias`)

Datos limpios y tipados correctamente, después de `spark_limpieza_datos_ventas.py`.


In [ ]:
df_silver = leer_parquet_minio(SILVER_BUCKET, SILVER_PREFIX, max_files=3)
print("Filas en la muestra de Silver:", len(df_silver))
display(df_silver.head())
display(df_silver.dtypes)

## 3. Muestra de Gold (`dataset_features`)

Dataset enriquecido con features de tiempo, lags y target (`target_ventas_proximo_dia`).


In [ ]:
df_gold = leer_parquet_minio(GOLD_BUCKET, GOLD_PREFIX_FEATURES, max_files=5)
print("Filas en la muestra de Gold (features):", len(df_gold))
display(df_gold.head())
display(df_gold.dtypes)

## 4. Calidad de datos básica en Gold

- Conteo de nulos en columnas clave.
- Rango de fechas disponible.


In [ ]:
columnas_clave = ["producto_id", "fecha", "cantidad_vendida", "ingreso_total"]

null_counts = df_gold[columnas_clave].isnull().sum()
display(null_counts.to_frame("nulos"))

print("Rango de fechas en la muestra de Gold:")
print("min:", df_gold["fecha"].min())
print("max:", df_gold["fecha"].max())

## 5. Ventas agregadas por día (muestra de Gold)

Evolución de unidades totales e ingreso por fecha.


In [ ]:
agg_por_dia = (
    df_gold
    .groupby("fecha", as_index=False)
    .agg(
        unidades_totales=("cantidad_vendida", "sum"),
        ingreso_total_dia=("ingreso_total", "sum"),
    )
    .sort_values("fecha")
)
display(agg_por_dia.head())

plt.figure(figsize=(10, 4))
plt.plot(agg_por_dia["fecha"], agg_por_dia["unidades_totales"], marker="o")
plt.xticks(rotation=45)
plt.title("Unidades vendidas por día (muestra Gold)")
plt.xlabel("Fecha")
plt.ylabel("Unidades")
plt.tight_layout()
plt.show()

## 6. Ventas por producto (muestra de Gold)

Ranking simple de productos por unidades vendidas.


In [ ]:
agg_por_producto = (
    df_gold
    .groupby("producto_id", as_index=False)
    .agg(
        unidades_totales=("cantidad_vendida", "sum"),
        ingreso_total_producto=("ingreso_total", "sum"),
    )
    .sort_values("unidades_totales", ascending=False)
)
display(agg_por_producto.head(20))

## 7. Estacionalidad simple (día de semana y mes)

Usamos las columnas de features generadas por Spark: `dia_semana`, `es_fin_semana`, `mes`.


In [ ]:
if {"dia_semana", "mes"}.issubset(df_gold.columns):
    por_dia_semana = (
        df_gold
        .groupby("dia_semana", as_index=False)
        .agg(unidades_promedio=("cantidad_vendida", "mean"))
        .sort_values("dia_semana")
    )
    display(por_dia_semana)

    por_mes = (
        df_gold
        .groupby("mes", as_index=False)
        .agg(unidades_promedio=("cantidad_vendida", "mean"))
        .sort_values("mes")
    )
    display(por_mes)
else:
    print("La muestra no contiene columnas de estacionalidad (dia_semana/mes)")